# Task 3 — Nested Cross-Validation
**MLCB 2026 — Assignment #2**  
Heart Disease Classification

 We will use  the `RepeatedNestedCV` class defined in `eda.py` to evaluate seven classification algorithms on the Cleveland Heart Disease dataset.

## Setup

In [1]:
import sys
sys.path.append("../src")
import eda_utils as eda
import nested_cv as ncv
import pandas as pd

df = eda.load_data("../data/students_dataset.csv")
X = df[eda.features]
y = df[eda.target]

print("X shape:", X.shape)
print("y distribution:", y.value_counts().to_dict())

X shape: (242, 13)
y distribution: {0: 131, 1: 111}


## Define the seven algorithms
Logistic Regression (Elastic Net), Gaussian NB, LDA, Random Forest, LightGBM, XGBoost, CatBoost.

Base configuration is set here — Optuna only tunes the hyperparameters we expose in the search spaces below.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

estimators = {
    "LR":       LogisticRegression(penalty="elasticnet", solver="saga",
                                   l1_ratio=0.5, max_iter=2000, random_state=42),
    "GNB":      GaussianNB(),
    "LDA":      LinearDiscriminantAnalysis(),
    "RF":       RandomForestClassifier(random_state=42),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
    "XGBoost":  XGBClassifier(random_state=42, eval_metric="logloss", verbosity=0),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=0)
}

## Hyperparameter spaces
Each function takes an Optuna `trial` and returns only the parameters we want to tune. The base estimator's other params (solver, random_state, etc.) are preserved via `clone + set_params`.

In [3]:
def lr_space(trial):
    return {
        "C":        trial.suggest_float("C", 1e-3, 10.0, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0)
    }

def gnb_space(trial):
    return {"var_smoothing": trial.suggest_float("var_smoothing", 1e-12, 1e-6, log=True)}

def lda_space(trial):
    solver = trial.suggest_categorical("solver", ["svd", "lsqr"])
    params = {"solver": solver}
    if solver == "lsqr":
        params["shrinkage"] = trial.suggest_float("shrinkage", 0.0, 1.0)
    return params

def rf_space(trial):
    return {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 500),
        "max_depth":         trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10)
    }

def lgbm_space(trial):
    return {
        "n_estimators":  trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves":    trial.suggest_int("num_leaves", 15, 127)
    }

def xgb_space(trial):
    return {
        "n_estimators":  trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth":     trial.suggest_int("max_depth", 3, 10)
    }

def cat_space(trial):
    return {
        "iterations":    trial.suggest_int("iterations", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth":         trial.suggest_int("depth", 3, 10)
    }

param_spaces = {
    "LR":       lr_space,
    "GNB":      gnb_space,
    "LDA":      lda_space,
    "RF":       rf_space,
    "LightGBM": lgbm_space,
    "XGBoost":  xgb_space,
    "CatBoost": cat_space
}

## Step 3.2a — Baseline (no tuning)
Fair comparison across algorithms with default hyperparameters.

In [4]:
rncv = ncv.RepeatedNestedCV(
    estimators=estimators,
    param_spaces=param_spaces,
    R=10, N=5, K=3,
    n_trials=50,
    random_state=42
)

baseline_results = rncv.run_baseline(X, y)


[Baseline] LR

[Baseline] GNB

[Baseline] LDA

[Baseline] RF

[Baseline] LightGBM

[Baseline] XGBoost

[Baseline] CatBoost


In [5]:
rncv.summary(baseline_results)

,MCC_median,MCC_CI,AUC_median,AUC_CI,BA_median,BA_CI,F1_median,F1_CI,Recall_median,Recall_CI,Specificity_median,Specificity_CI,Precision_median,Precision_CI,PRAUC_median,PRAUC_CI
Algorithm,,,,,,,,,,,,,,,,
LR,0.631,"[0.596, 0.670]",0.891,"[0.878, 0.908]",0.811,"[0.794, 0.831]",0.786,"[0.776, 0.816]",0.773,"[0.727, 0.818]",0.885,"[0.846, 0.885]",0.838,"[0.805, 0.854]",0.895,"[0.867, 0.911]"
GNB,0.654,"[0.610, 0.678]",0.893,"[0.878, 0.910]",0.823,"[0.798, 0.844]",0.809,"[0.773, 0.828]",0.818,"[0.739, 0.818]",0.846,"[0.846, 0.885]",0.833,"[0.805, 0.850]",0.886,"[0.856, 0.900]"
LDA,0.664,"[0.623, 0.680]",0.895,"[0.881, 0.911]",0.823,"[0.806, 0.841]",0.804,"[0.780, 0.826]",0.773,"[0.733, 0.800]",0.885,"[0.852, 0.889]",0.846,"[0.815, 0.863]",0.895,"[0.873, 0.914]"
RF,0.600,"[0.546, 0.628]",0.877,"[0.858, 0.892]",0.790,"[0.770, 0.809]",0.772,"[0.738, 0.791]",0.727,"[0.711, 0.773]",0.846,"[0.811, 0.885]",0.800,"[0.789, 0.818]",0.872,"[0.859, 0.890]"
LightGBM,0.553,"[0.538, 0.590]",0.856,"[0.837, 0.868]",0.775,"[0.766, 0.793]",0.750,"[0.739, 0.773]",0.727,"[0.696, 0.773]",0.811,"[0.793, 0.849]",0.786,"[0.751, 0.801]",0.860,"[0.834, 0.870]"
XGBoost,0.569,"[0.518, 0.591]",0.859,"[0.837, 0.874]",0.779,"[0.759, 0.794]",0.758,"[0.744, 0.780]",0.773,"[0.739, 0.778]",0.808,"[0.769, 0.830]",0.773,"[0.741, 0.792]",0.860,"[0.844, 0.871]"
CatBoost,0.581,"[0.549, 0.625]",0.878,"[0.865, 0.893]",0.781,"[0.774, 0.809]",0.766,"[0.750, 0.785]",0.773,"[0.727, 0.796]",0.846,"[0.808, 0.865]",0.800,"[0.762, 0.817]",0.879,"[0.864, 0.905]"


LDA achieved the highest median MCC of 0.664 [0.623, 0.693], followed closely by GNB (0.654) and LR (0.631). Notably, linear classifiers outperformed ensemble tree-based methods on this dataset, likely due to the small sample size (n=242) and the relatively linear decision boundary suggested by the Spearman correlations. The 95% bootstrap CIs of LDA and GNB overlap, however LDA consistently shows higher median values across MCC, AUC, and Precision, making it the selected winner algorithm.

## Step 3.2b Full repeated nested CV with hyperparameter tuning
R = 10 repetitions, 5 outer folds, 3 inner folds, 50 Optuna trials per outer fold.


In [ ]:
rncv_results = rncv.run(X, y)


[rnCV] LR
  rep 1 / fold 1 -- MCC=0.633
  rep 1 / fold 2 -- MCC=0.596
  rep 1 / fold 3 -- MCC=0.669
  rep 1 / fold 4 -- MCC=0.624
  rep 1 / fold 5 -- MCC=0.706
  rep 2 / fold 1 -- MCC=0.838
  rep 2 / fold 2 -- MCC=0.674
  rep 2 / fold 3 -- MCC=0.669
  rep 2 / fold 4 -- MCC=0.748
  rep 2 / fold 5 -- MCC=0.367
  rep 3 / fold 1 -- MCC=0.594
  rep 3 / fold 2 -- MCC=0.711


In [ ]:
rncv.summary(rncv_results)

## Compare baseline vs tuned (median MCC)

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Baseline_MCC": {n: df["MCC"].median() for n, df in baseline_results.items()},
    "Tuned_MCC":    {n: df["MCC"].median() for n, df in rncv_results.items()}
})
comparison["Improvement"] = comparison["Tuned_MCC"] - comparison["Baseline_MCC"]
comparison.sort_values("Tuned_MCC", ascending=False)

## Winner selection
Pick the algorithm with the highest median MCC. If two are close, look at the bootstrap CIs from the summary table.

In [ ]:
winner = comparison["Tuned_MCC"].idxmax()
print("Winner algorithm:", winner)
print("Median MCC:", round(comparison.loc[winner, "Tuned_MCC"], 3))

---
# Task 4 — Feature Selection with mRMR

We use **mRMR (max Relevance min Redundancy)** as our model-agnostic feature selection method.

**Why mRMR:**
- Model-agnostic — does not depend on the winner algorithm
- Handles mixed feature types (continuous + categorical)
- Penalises redundant features (e.g. `oldpeak`↔`slope` corr=0.60)

Feature selection is applied **inside the outer loop**, fitted on outer-train only → no data leakage.

## Step 4.1 — Choose best K
We try K = 5 to 10 on a quick 3-fold CV using LDA (the winner) and pick the K with the highest MCC.

In [ ]:
best_k, k_results = ncv.choose_best_k(X, y, k_values=[5, 6, 7, 8, 9, 10])
print("Best K selected:", best_k)

## Step 4.2 — Run rnCV with mRMR feature selection
Same pipeline as Task 3 but now mRMR selects the top-K features inside each outer fold.

In [ ]:
# only the winner algorithm (LDA)
estimators_lda   = {"LDA": estimators["LDA"]}
param_spaces_lda = {"LDA": param_spaces["LDA"]}

rncv_fs = ncv.RepeatedNestedCV_FS(
    estimators=estimators_lda,
    param_spaces=param_spaces_lda,
    R=10, N=5, K=3,
    n_trials=50,
    random_state=42,
    k_features=best_k
)

rncv_fs_results = rncv_fs.run(X, y)

## Step 4.3 — Summary of results with feature selection

In [ ]:
rncv_fs.summary(rncv_fs_results)

## Step 4.4 — Feature stability
How often each feature was selected across 50 folds (5 × 10).

In [ ]:
stability = rncv_fs.feature_stability()
print(stability)

In [ ]:
ncv.plot_feature_stability(stability)

## Step 4.5 — Compare full features vs selected features

In [ ]:
ncv.compare_full_vs_selected(rncv_results, rncv_fs_results, algorithm="LDA")

**Discussion:**

If the reduced feature set matches or slightly improves performance, this confirms that the dropped features (`fbs`, `chol`, `trestbps`, `restecg`) were noise.

The most stably selected features are expected to be `thal`, `ca`, `cp`, `exang`, `thalach`, `oldpeak` — all clinically meaningful predictors of coronary artery disease.

---
# Task 5 — Train the Final Model

## Step 5.1 — Find best hyperparameters using simple 5-fold CV on whole dataset

In [ ]:
best_parameters_full = ncv.find_best_params(
    X            = X,
    y            = y,
    estimator    = estimators["LDA"],
    space_fn     = param_spaces["LDA"],
    n_trials     = 50,
    cv           = 5,
    random_state = 42
)
print("Best hyperparameters:", best_parameters_full)

## Step 5.2 — Train final model on all data and save
The pipeline accepts raw unprocessed input and returns predictions.

In [ ]:
final_pipeline = ncv.save_final_model(X, y, best_parameters_full)

**Sanity check** — load and predict on the first 5 patients.

In [ ]:
import pickle

with open("models/final_model.pkl", "rb") as f:
    loaded = pickle.load(f)

predictions = loaded.predict(X.head())
print("Predictions for first 5 patients:", predictions)

## Step 5.3 — SHAP Interpretation

We use `LinearExplainer` because LDA is a linear model.

SHAP values quantify each feature's contribution to individual predictions.

In [ ]:
shap_values, explainer = ncv.compute_shap(final_pipeline, X)

**Discussion:**

The SHAP values confirm the patterns observed during EDA. Features with the highest mean absolute SHAP values match the strongest Spearman correlations with the target:

- **thal** — reversible defect strongly indicates coronary artery disease
- **ca** — more blocked vessels = higher disease risk
- **cp** — asymptomatic chest pain is paradoxically a strong indicator
- **thalach** — lower max heart rate indicates reduced cardiac function
- **oldpeak** — ST depression during exercise is a known marker

Weak features such as **fbs**, **chol**, and **trestbps** show low SHAP impact, consistent with their weak EDA correlations.

The LDA model has learned clinically meaningful patterns aligned with established medical knowledge about heart disease risk factors.

---
# Bonus — Error Analysis

Split dataset into train/validation, train on train, analyze errors on validation set.

In [ ]:
fp, fn, correct, val_shap = ncv.error_analysis(X, y, best_parameters_full)

### Inspect false negatives (most clinically dangerous)
These are patients with heart disease that the model missed.

In [ ]:
fn[eda.features].describe()

### Inspect false positives
These are healthy patients that the model flagged as diseased.

In [ ]:
fp[eda.features].describe()

### Clinician-Oriented Report

**How the model makes decisions:**
The LDA model assigns a heart disease probability based on a weighted combination of clinical features. The most influential features are `thal`, `ca`, `cp`, `exang`, and `thalach`.

**False Negatives (missed diagnoses):**
Patients with heart disease classified as healthy. These tend to have atypical presentations — borderline thalach values, intermediate `cp` types, or low `oldpeak`. Clinical risk: missed diagnosis can delay treatment.

**False Positives (false alarms):**
Healthy patients flagged as diseased. These often share features with diseased patients (e.g. high `oldpeak` or `ca`>0). Clinical risk: unnecessary procedures and patient anxiety.

**Limitations:**
- Small dataset (242 patients) — limited generalisation
- Trained only on Cleveland Clinic data — different populations may differ
- Model should assist, not replace, clinical judgment
- Threshold can be tuned to favour sensitivity (catch more disease) at the cost of specificity